## OpenAI (ChatGPT)

Beispiel für die Erstellung einer Python Applikation

Zuerst muss der Open AI API Key erstellt und ein Guthaben bereitgestellt werden:
* [API Key erstellen](https://platform.openai.com/account/api-keys)
* [Guthaben bereitstellen](https://platform.openai.com/settings/organization/billing/overview)


In [ ]:
import os
from openai import OpenAI
import json
from nbconvert import MarkdownExporter
from nbconvert.preprocessors import ExtractOutputPreprocessor
from IPython.display import Markdown, display
from datetime import datetime

os.environ['OPENAI_API_KEY']=''

Um den Prozess zu automatisieren, erstellen wir eine Funktion, die eine neue Benutzeranfrage entgegennimmt, sie an die KI sendet und die Antwort zurückgibt.

Allgemeine Anweisungen, wie Output im Markdown-Format etc., geben wir in `messages` mit.

In [ ]:
# OpenAI API-Schlüssel einrichten
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# 1. Vektor-Store erstellen
try:
    vector_store = client.vector_stores.create(name="cna_vector_store")
    vector_store_id = vector_store.id
    print(f"Vektor-Store mit ID '{vector_store_id}' erstellt.")
except Exception as e:
    print(f"Fehler beim Erstellen des Vektor-Stores: {e}")
    exit()

# 2. Dateien in 'cna' und Unterverzeichnissen suchen und hochladen
cna_directory = "cna"  # Pfad zum Verzeichnis, das die Dateien enthält
supported_extensions = [".md", ".pdf", ".ipynb"]

for root, _, files in os.walk(cna_directory):
    for filename in files:
        filepath = os.path.join(root, filename)
        file_extension = os.path.splitext(filename)[1]

        if file_extension in supported_extensions:
            try:
                if file_extension == ".ipynb":
                    # Jupyter Notebook in Markdown konvertieren
                    exporter = MarkdownExporter()
                    exporter.register_preprocessor(
                        ExtractOutputPreprocessor(output_dir='output_files'),
                        enabled=True
                    )
                    body, resources = exporter.from_filename(filepath)
                    markdown_filepath = os.path.splitext(filepath)[0] + ".md"
                    with open(markdown_filepath, "w", encoding="utf-8") as md_file:
                        md_file.write(body)

                    # Hochladen & zum Vector Store hinzufügen
                    with open(markdown_filepath, "rb") as file:
                        uploaded_file = client.files.create(file=file, purpose='assistants')
                        client.vector_stores.files.create(
                            vector_store_id=vector_store_id,
                            file_id=uploaded_file.id
                        )
                        print(f"Notebook '{filepath}' konvertiert und hochgeladen als '{markdown_filepath}'.")

                    # Optional: temporäre Datei entfernen
                    os.remove(markdown_filepath)

                else:
                    with open(filepath, "rb") as file:
                        uploaded_file = client.files.create(file=file, purpose='assistants')
                        client.vector_stores.files.create(
                            vector_store_id=vector_store_id,
                            file_id=uploaded_file.id
                        )
                        print(f"Datei '{filepath}' erfolgreich hochgeladen und zum Vector Store hinzugefügt.")

            except Exception as e:
                print(f"Fehler beim Verarbeiten der Datei '{filepath}': {e}")

print("✅ Fertig mit Hochladen und Zuordnen zum Vector Store.")


Anzeige der Vector Stores

In [ ]:
import os
from openai import OpenAI

# OpenAI API-Schlüssel einrichten
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

try:
    # Alle Vektor-Stores auflisten
    response = client.vector_stores.list()
    vector_stores = response.data

    if vector_stores:
        print("Deine vorhandenen Vector Stores:")
        for vs in vector_stores:
            print(f"- ID: {vs.id}, Name: {vs.name}, Created At: {vs.created_at}")
    else:
        print("Es wurden keine Vector Stores gefunden.")

except Exception as e:
    print(f"Fehler beim Auflisten der Vector Stores: {e}")
    
vs = client.vector_stores.retrieve("")
print(f"Dokumente im Vector Store: {vs.file_counts}")
    

print("Fertig.")

### Abfrage

In [ ]:
# ID deines spezifischen Vector Stores
vector_store_id = ""  # Ersetze dies mit deiner tatsächlichen Vector Store ID

def query_vector_store(query, top_k=3):
    """Führt eine semantische Suche im Vector Store durch."""
    try:
        response = client.responses.create(
            model="gpt-4o-mini",
            input=query,
            tools=[{
                "type": "file_search",
                "vector_store_ids": [vector_store_id]
            }]
        )
        return response
    except Exception as e:
        print(f"Fehler bei der Suche im Vector Store: {e}")
        return None

# Starte eine Konversation
messages = [
    {"role": "system", "content": "Du bist ein hilfsbereiter Assistent."}
]



In [ ]:
user_input = "wie installiere ich istio?"

# Führe eine Suche im Vector Store durch
response = query_vector_store(user_input)

# Extrahiere den Markdown-Text aus der Antwort
if response:
    # response.output enthält eine Liste mit Nachrichten
    for item in response.output:
        if item.type == "message" and hasattr(item, "content"):
            for content_item in item.content:
                if content_item.type == "output_text" and hasattr(content_item, "text"):
                    markdown_text = content_item.text
                    display(Markdown(markdown_text))


---

### Debug 

In [ ]:
import os
import json
from openai import OpenAI
from IPython.display import Markdown, display

# OpenAI-Client initialisieren
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ID deines spezifischen Vector Stores
vector_store_id = ""

def query_vector_store(query, top_k=3):
    """Führt eine semantische Suche im Vector Store durch."""
    try:
        response = client.responses.create(
            model="gpt-4o-mini",
            input=query,
            tools=[{
                "type": "file_search",
                "vector_store_ids": [vector_store_id],
                "max_num_results": top_k
            }]
        )
        return response
    except Exception as e:
        print(f"Fehler bei der Suche im Vector Store: {e}")
        return None

# Benutzeranfrage
user_input = "wie installiere ich istio und mit welchen zusatztools und wie greife ich auf die port zu?"

# Vektor-Suche durchführen
response = query_vector_store(user_input)

# Formatiertes JSON anzeigen (z. B. zum Debuggen)
if response:
    # Versuche, das Response-Objekt in ein dict zu konvertieren, falls nötig
    if hasattr(response, 'model_dump'):
        response_dict = response.model_dump()  # bei pydantic-basierten Objekten
    else:
        response_dict = response  # falls es schon ein dict ist

    # Formatiert anzeigen
    print(json.dumps(response_dict, indent=2, ensure_ascii=False))


- - -

### Aufräumen 

löschen aller Vector Stores!

In [ ]:
import os
from openai import OpenAI

# OpenAI API-Schlüssel einrichten
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

try:
    # Alle Vektor-Stores auflisten
    response = client.vector_stores.list()
    vector_stores = response.data

    if vector_stores:
        print("Folgende Vector Stores werden gelöscht:")
        for vs in vector_stores:
            print(f"- ID: {vs.id}, Name: {vs.name}")

        confirmation = input("Bist du sicher, dass du ALLE diese Vector Stores löschen möchtest? (ja/nein): ")

        if confirmation.lower() == "ja":
            for vs in vector_stores:
                try:
                    response = client.vector_stores.delete(vector_store_id=vs.id)
                    if response.deleted:
                        print(f"Vector Store mit ID '{vs.id}' erfolgreich gelöscht.")
                    else:
                        print(f"Fehler beim Löschen des Vector Stores mit ID '{vs.id}'.")
                except Exception as e:
                    print(f"Fehler beim Löschen des Vector Stores mit ID '{vs.id}': {e}")
            print("Alle Vector Stores wurden (versucht zu) löschen.")
        else:
            print("Löschvorgang abgebrochen.")
    else:
        print("Es wurden keine Vector Stores gefunden, die gelöscht werden könnten.")

except Exception as e:
    print(f"Fehler beim Auflisten der Vector Stores: {e}")

print("Fertig.")